# Question tree: results

Apertus v1.5 labels product images without any finetuning. This notebook shows how the method was found and what
each step is worth. Every number below is computed here, from the stored answers of Apertus and the human labels in
`train.csv`, with the leaderboard metric. Nothing is sent to the model.

The answers were collected with `question_tree.ask` and `question_tree.experiments`. They are stored in
`runs/question_tree`, next to the competition data and outside the repository.

In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score

from question_tree import ask, combiner, contours, data, evaluate, pixels, tree
from question_tree.config import CLASSES, DATA_DIR, DIMENSIONS, MODEL_8B, MODEL_70B, RUNS_DIR, SEED, STORE
from question_tree.images import load_rgb
from question_tree.questions import ASKED_ALONE, ASKED_TOGETHER, TREE_MULTIPLE_CHOICE, TREE_QUESTIONS, TREE_YES_NO

pd.set_option("display.max_columns", 30, "display.width", 220, "display.max_colwidth", 90)

train = data.load_train()
sets = data.image_sets(train)
answers = {name: ask.read_answers(STORE / f"answers_{name}_{MODEL_8B}.jsonl").reindex(rows["image_id"]) for name, rows in sets.items()}
backdrops = {name: evaluate.measure_backdrops(rows) for name, rows in sets.items()}
baseline = evaluate.read_baseline(MODEL_8B, RUNS_DIR)

## 1. The image sets

All images are train images, because only they have labels. The holdout is 20% of the products. Everything was
designed on the tuning images, which lie outside the holdout and share no product with the other two sets. Those two
were scored only after the tree was frozen, so they show whether a result holds on unseen images. They are drawn
image by image from the same holdout, so they share no image but some products.

In [2]:
holdout = data.holdout_mask(train)
overview = pd.DataFrame({
    name: {
        "images": len(rows),
        "products": rows["product_id"].nunique(),
        "inside the holdout": int(rows["image_id"].isin(train[holdout]["image_id"]).sum()),
        "scored cells": int(sum(data.is_scored(rows, d).sum() for d in DIMENSIONS)),
    }
    for name, rows in sets.items()
}).T
names = list(sets)
shared = {f"{a} and {b}": len(set(sets[a]["product_id"]) & set(sets[b]["product_id"])) for i, a in enumerate(names) for b in names[i + 1:]}
print("products shared between two sets:", shared)
overview

products shared between two sets: {'tune and test': 0, 'tune and benchmark': 0, 'test and benchmark': 76}


,images,products,inside the holdout,scored cells
tune,474,448,0,2232
test,300,271,300,1361
benchmark,500,414,500,2297


## 2. The starting point: one prompt for all six labels

The baseline of `apertus_zero_shot_api.ipynb` asks for all six labels in one request. The table shows, per dimension,
the single answer the model gives most often on the benchmark images, and how large its share is. The model sees the
image, but it does not decide.

In [3]:
rows = sets["benchmark"]
given = baseline.set_index("image_id").loc[rows["image_id"]]
collapse = {}
for dimension in DIMENSIONS:
    scored = data.is_scored(rows, dimension).to_numpy()
    counts = given[dimension][scored].value_counts()
    collapse[dimension] = {
        "scored cells": int(scored.sum()),
        "classes in the labels": rows[dimension][scored].nunique(),
        "most frequent answer": counts.index[0],
        "its share of the answers": round(counts.iloc[0] / scored.sum(), 3),
    }
display(pd.DataFrame(collapse).T)
evaluate.score_table({"constant answer": evaluate.constant_predictions(rows), "one prompt for all six labels": baseline}, rows)

,scored cells,classes in the labels,most frequent answer,its share of the answers
subject,474,13,full_product,1.0
background,413,3,white,0.77
perspective,328,10,front_head_on,0.872
individuals,413,2,none,0.998
text_classification,459,6,none,0.978
text_language,210,7,none,0.543


,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
constant answer,500,0.058,0.308,0.047,0.489,0.152,0.072,0.188,0.592
one prompt for all six labels,500,0.058,0.545,0.070,0.546,0.150,0.352,0.287,0.634


## 3. Small questions

The same model answers a plain yes/no question correctly where the one-prompt answer fails. The first table counts,
on the benchmark images, how many images of a class each way of asking finds, and how often the question says yes
where it should not.

In [4]:
CLEAR_CASES = {  # class: (dimension, yes/no question that asks for it)
    "energy_label": ("subject", "energy_label"), "logo": ("subject", "logo2"), "placeholder": ("subject", "placeholder"),
    "individual_present": ("individuals", "person"), "nutritional_info": ("text_classification", "nutrition"),
}
rows, table = sets["benchmark"], {}
for name, (dimension, key) in CLEAR_CASES.items():
    scored = data.is_scored(rows, dimension).to_numpy()
    is_class = (rows[dimension] == name).to_numpy() & scored
    said_yes = (answers["benchmark"][key] == "yes").to_numpy()
    table[name] = {
        "images of this class": int(is_class.sum()),
        "found by the one prompt": int((given[dimension].to_numpy() == name)[is_class].sum()),
        "found by the yes/no question": int(said_yes[is_class].sum()),
        "false alarms of the question": int(said_yes[scored & ~is_class].sum()),
        "other images": int((scored & ~is_class).sum()),
    }
pd.DataFrame(table).T

,images of this class,found by the one prompt,found by the yes/no question,false alarms of the question,other images
energy_label,22,0,21,0,452
logo,14,0,13,12,460
placeholder,5,0,4,4,469
individual_present,17,1,16,4,396
nutritional_info,25,0,16,2,434


Every yes/no question of the tree, on the tuning images. "Should say yes" counts the images whose human label makes
yes the right answer. Some questions are reliable, others are weak, and the tree only leans on a weak one after the
reliable ones have had their say.

In [5]:
MULTI = ["multipack_trade_unit", "delivery_scope", "product_grouping", "multiview"]
NO_PHOTO = ["infographic", "logo", "placeholder", "energy_label"]
RIGHT_ANSWER_IS_YES = {  # question: (dimension, classes for which yes is right)
    "energy_label": ("subject", ["energy_label"]), "placeholder": ("subject", ["placeholder"]), "logo2": ("subject", ["logo"]),
    "photo": ("subject", [c for c in CLASSES["subject"] if c not in NO_PHOTO]), "infographic": ("subject", ["infographic"]),
    "textonly": ("subject", ["infographic"]), "pack_and_real": ("subject", ["package_and_product"]), "box_only": ("subject", ["packaging"]),
    "packaging": ("subject", ["packaging", "package_and_product"]), "variants": ("subject", ["product_grouping"]),
    "identical": ("subject", ["multipack_trade_unit"]), "several": ("subject", MULTI), "scope": ("subject", ["delivery_scope"]),
    "scene": ("background", ["scene_lifestyle"]), "plain": ("background", ["white", "color"]), "person": ("individuals", ["individual_present"]),
    "nutrition": ("text_classification", ["nutritional_info"]), "dims": ("text_classification", ["technical_specs"]),
    "promo": ("text_classification", ["promotional_text"]), "seal": ("text_classification", ["certification_seal"]),
    "words2": ("text_language", [c for c in CLASSES["text_language"] if c != "none"]),
}


def question_quality(name):
    rows, table = sets[name], {}
    for key, (dimension, classes) in RIGHT_ANSWER_IS_YES.items():
        scored = data.is_scored(rows, dimension).to_numpy()
        should = rows[dimension].isin(classes).to_numpy()
        said = (answers[name][key] == "yes").to_numpy()
        positives, negatives = int((scored & should).sum()), int((scored & ~should).sum())
        table[key] = {
            "should say yes": positives, "said yes": int((said & scored & should).sum()),
            "should say no": negatives, "said no": int((~said & scored & ~should).sum()),
        }
        table[key]["balanced accuracy"] = round((table[key]["said yes"] / positives + table[key]["said no"] / negatives) / 2, 2)
    return pd.DataFrame(table).T.astype({column: int for column in ("should say yes", "said yes", "should say no", "said no")})


question_quality("tune")

,should say yes,said yes,should say no,said no,balanced accuracy
energy_label,12,12,419,419,1.00
placeholder,12,12,419,416,1.00
logo2,12,12,419,405,0.98
photo,359,281,72,68,0.86
infographic,36,21,395,344,0.73
textonly,36,13,395,367,0.65
pack_and_real,23,17,408,394,0.85
box_only,27,13,404,388,0.72
packaging,50,20,381,377,0.69
variants,13,9,418,412,0.84


## 4. The decision tree

`question_tree.tree.classify` turns the answers of one image into its labels. The table compares, on each image set,
the constant answer, the one-prompt baseline, one multiple-choice question per dimension, and the tree in its first
version and with the sharpened rules of section 6. The tuning images are class balanced and were used for the design,
so only the other two sets say how the tree does on unseen images.

In [6]:
def methods(name, with_baseline=True):
    rows = sets[name]
    result = {"constant answer": evaluate.constant_predictions(rows)}
    if with_baseline:
        result["one prompt for all six labels"] = baseline
    result["one multiple-choice question per dimension"] = evaluate.multiple_choice_predictions(answers[name])
    result["tree, first version"] = evaluate.tree_predictions(answers[name], backdrops[name], tree.NO_RULES)
    result["tree with sharpened rules"] = evaluate.tree_predictions(answers[name], backdrops[name])
    return result


for name in sets:
    print(f"\n{name}")
    display(evaluate.score_table(methods(name), sets[name]))


tune


,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
constant answer,474,0.051,0.287,0.047,0.477,0.142,0.063,0.178,0.494
one prompt for all six labels,474,0.051,0.544,0.104,0.605,0.158,0.465,0.321,0.569
one multiple-choice question per dimension,474,0.411,0.511,0.107,0.968,0.482,0.554,0.505,0.649
"tree, first version",474,0.486,0.768,0.159,0.968,0.499,0.554,0.572,0.695
tree with sharpened rules,474,0.546,0.768,0.159,0.968,0.525,0.590,0.593,0.718



test


,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
constant answer,300,0.056,0.302,0.042,0.481,0.185,0.102,0.195,0.578
one prompt for all six labels,300,0.056,0.454,0.095,0.535,0.185,0.600,0.321,0.631
one multiple-choice question per dimension,300,0.417,0.494,0.132,0.927,0.410,0.611,0.498,0.719
"tree, first version",300,0.431,0.770,0.146,0.927,0.447,0.611,0.555,0.747
tree with sharpened rules,300,0.432,0.770,0.146,0.927,0.446,0.635,0.559,0.751



benchmark


,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
constant answer,500,0.058,0.308,0.047,0.489,0.152,0.072,0.188,0.592
one prompt for all six labels,500,0.058,0.545,0.070,0.546,0.150,0.352,0.287,0.634
one multiple-choice question per dimension,500,0.389,0.544,0.112,0.929,0.484,0.484,0.490,0.717
"tree, first version",500,0.431,0.733,0.151,0.929,0.486,0.484,0.536,0.743
tree with sharpened rules,500,0.483,0.733,0.151,0.929,0.476,0.508,0.547,0.754


Per class of the subject dimension on the benchmark images: the tree against the one-prompt baseline.

In [7]:
rows = sets["benchmark"]
before = evaluate.evaluate(baseline, rows)["subject"]["per_class"]
after = evaluate.evaluate(methods("benchmark")["tree with sharpened rules"], rows)["subject"]["per_class"]
pd.DataFrame({
    name: {"scored cells": values["support"], "F1 one prompt": round(before.get(name, {"f1": 0})["f1"], 2), "F1 tree": round(values["f1"], 2),
           "precision tree": round(values["precision"], 2), "recall tree": round(values["recall"], 2)}
    for name, values in sorted(after.items(), key=lambda item: -item[1]["support"])
}).T.astype({"scored cells": int})

,scored cells,F1 one prompt,F1 tree,precision tree,recall tree
full_product,285,0.75,0.80,0.87,0.73
infographic,46,0.00,0.65,0.63,0.67
detail_closeup,34,0.00,0.42,0.71,0.29
energy_label,22,0.00,0.98,1.00,0.95
delivery_scope,18,0.00,0.44,0.32,0.72
package_and_product,15,0.00,0.75,0.71,0.80
packaging,15,0.00,0.19,0.14,0.33
logo,14,0.00,0.74,0.62,0.93
multiview,7,0.00,0.00,0.00,0.00
placeholder,5,0.00,0.62,0.50,0.80


## 5. Which questions carry the result

One question at a time is switched off, by replacing its answers with the neutral answer, and the tree is scored
again on the benchmark images. The drop of the leaderboard score shows how much the tree relies on that question.

In [8]:
NEUTRAL = {"subject_mc": "full_product", "overlay_mc": "none", "lang": "none", "photo": "yes", "plain": "yes"}
rows, name = sets["benchmark"], "benchmark"
full = evaluate.evaluate(evaluate.tree_predictions(answers[name], backdrops[name]), rows)
impact = {}
for key in TREE_QUESTIONS:
    without = answers[name].copy()
    without[key] = NEUTRAL.get(key, "no")
    result = evaluate.evaluate(evaluate.tree_predictions(without, backdrops[name]), rows)
    changes = {d: result[d]["f1_macro"] - full[d]["f1_macro"] for d in DIMENSIONS}
    hit = min(changes, key=changes.get)
    impact[key] = {"change of the overall score": round(result["overall"]["f1_macro"] - full["overall"]["f1_macro"], 3), "dimension hit most": hit, "change there": round(changes[hit], 3)}
pd.DataFrame(impact).T.sort_values("change of the overall score").head(12)

,change of the overall score,dimension hit most,change there
lang,-0.073,text_language,-0.435
words2,-0.073,text_language,-0.435
person,-0.073,individuals,-0.44
nutrition,-0.022,text_classification,-0.13
dims,-0.021,text_classification,-0.129
photo,-0.018,subject,-0.106
energy_label,-0.014,subject,-0.084
overlay_mc,-0.011,text_classification,-0.066
pack_and_real,-0.01,subject,-0.062
logo2,-0.01,subject,-0.062


## 6. Sharper rules from the failed cases

Eleven sharper rules were written after looking at tuning images that the first tree got wrong. Each rule is a switch
in `tree.classify`. The table shows what each rule alone changes in the leaderboard score. A rule was kept when it
gained more than 0.001 on the tuning images. The other two columns show whether that gain holds on unseen images.

In [9]:
def overall(name, rules):
    return evaluate.evaluate(evaluate.tree_predictions(answers[name], backdrops[name], frozenset(rules)), sets[name])["overall"]["f1_macro"]


first = {name: overall(name, tree.NO_RULES) for name in sets}
effects = pd.DataFrame({
    rule: {"rule": text, **{name: round(overall(name, {rule}) - first[name], 3) for name in sets}, "kept": rule in tree.KEPT_RULES}
    for rule, text in tree.SHARPENED_RULES.items()
}).T
display(effects)
pd.DataFrame({
    "tree, first version": first,
    "with the kept rules": {name: overall(name, tree.KEPT_RULES) for name in sets},
    "with all eleven rules": {name: overall(name, tree.SHARPENED_RULES) for name in sets},
}).round(3)

,rule,tune,test,benchmark,kept
no_fallback_to_multiple_choice,pack-plus-product and multiview are no longer taken from the multiple-choice answer,0.004,0.004,0.003,True
seal_is_not_a_logo,"logo: seals, eco-labels and icons are not brand logos",0.002,-0.0,0.0,True
lone_seal_or_text_is_infographic,infographic: also a seal on its own or an image that is only text,0.002,0.0,0.001,True
printed_product_is_packaging,"packaging: a closed box, also with the product printed on it; pack-plus-product needs ...",0.003,-0.001,0.003,True
food_pack_is_the_product,"food, drinks and supplements in their own pack are the product",0.001,0.001,0.001,False
scope_needs_loose_items,delivery scope: the main product plus at least two loose items,0.001,0.004,0.006,False
multiview_is_the_same_item,multiview: the very same item shown twice,-0.002,-0.0,-0.001,False
strict_nutrition_table,"nutrition: a table with energy, fat, carbohydrates and protein as numbers",0.001,-0.004,-0.002,False
lone_seal_is_certification,a seal on its own is the overlay class certification_seal,0.004,-0.0,-0.002,True
promotion_is_an_added_layer,"promotional text must be an added layer, not printed on the pack",-0.0,0.019,-0.005,False


,"tree, first version",with the kept rules,with all eleven rules
tune,0.572,0.593,0.593
test,0.555,0.559,0.559
benchmark,0.536,0.547,0.544


Sharper wording was also tried inside the multiple-choice questions, as longer explanations per class. On the tuning
images that lowers the score.

In [10]:
name = "tune"
short = evaluate.multiple_choice_predictions(answers[name])
sharpened = evaluate.multiple_choice_predictions(answers[name].assign(subject_mc=answers[name]["subject_mc2"], overlay_mc=answers[name]["overlay_mc2"]))
evaluate.score_table({"multiple choice, short explanations": short, "multiple choice, sharpened explanations": sharpened}, sets[name])

,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
"multiple choice, short explanations",474,0.411,0.511,0.107,0.968,0.482,0.554,0.505,0.649
"multiple choice, sharpened explanations",474,0.393,0.511,0.107,0.968,0.304,0.554,0.473,0.650


## 7. White backdrop: measured, not asked

The guideline defines white as brightness 250 or more. `question_tree.pixels` reads the four corner patches of the
image and calls the backdrop white when at least two of them reach 250. The first table scores the background
dimension on the benchmark images for three ways of deciding it.

A frame around the image would spoil a measurement taken right at the border. The second table draws frames of
different widths around the tuning images that have a human white or color label and checks the rule again, once
measured only at the border and once also a few steps inside it. The pipeline uses the second version: it holds up
under frames, and on the benchmark images the two versions differ by a few thousandths.

In [11]:
name, rows = "benchmark", sets["benchmark"]
asked = evaluate.multiple_choice_predictions(answers[name])
at_border = evaluate.tree_predictions(answers[name], backdrops[name], white_column="white_at_border")
stepping_in = evaluate.tree_predictions(answers[name], backdrops[name])
print("background, macro F1 on the benchmark images")
display(evaluate.score_table({
    "one prompt for all six labels": baseline, "asked as a multiple-choice question": asked,
    "scene asked, white measured at the border": at_border, "scene asked, white measured with frame tolerance": stepping_in,
}, rows)[["images", "background"]])

plain = sets["tune"][data.is_scored(sets["tune"], "background") & (sets["tune"]["background"] != "scene_lifestyle")]
grays = [pixels.brightness(load_rgb(DATA_DIR / path)) for path in plain["image_path"]]
is_white = (plain["background"] == "white").to_numpy()
frames = {"no frame": (0, 0), "1 pixel, black": (1, 0), "3 pixels, black": (3, 0), "10 pixels, black": (10, 0), "1% of the short side, grey": (None, 128)}
check = {}
for label, (width, value) in frames.items():
    framed = [gray if width == 0 else pixels.with_frame(gray, width or max(1, int(0.01 * min(gray.shape))), value) for gray in grays]
    check[label] = {
        "measured at the border": round(float(np.mean([pixels.corners_are_white(g) for g in framed] == is_white)), 3),
        "also measured inside": round(float(np.mean([any(pixels.corners_are_white(g, step) for step in pixels.STEPS_INSIDE) for g in framed] == is_white)), 3),
    }
print(f"\nshare of correct white or color decisions on {len(plain)} tuning images ({int(is_white.sum())} white, {int((~is_white).sum())} color)")
pd.DataFrame(check).T

background, macro F1 on the benchmark images


,images,background
one prompt for all six labels,500,0.545
asked as a multiple-choice question,500,0.544
"scene asked, white measured at the border",500,0.736
"scene asked, white measured with frame tolerance",500,0.733



share of correct white or color decisions on 360 tuning images (304 white, 56 color)


,measured at the border,also measured inside
no frame,0.908,0.917
"1 pixel, black",0.869,0.906
"3 pixels, black",0.764,0.883
"10 pixels, black",0.336,0.814
"1% of the short side, grey",0.244,0.878


## 8. Perspective from contours

Apertus cannot tell from which side a product is seen: with the tree, perspective stays close to the constant answer.
`question_tree.contours` describes each image by its silhouette, the directions of its edges, a coarse brightness grid
and its shadows, and a tree ensemble learns the perspective from these numbers. It is trained on the labeled train
images outside the holdout and tested on the holdout, so no product is in both.

In [12]:
labeled = train[data.is_scored(train, "perspective")]
in_holdout = data.holdout_mask(labeled).to_numpy()
fit_rows, test_rows = labeled[~in_holdout], labeled[in_holdout]
fit_features, test_features = contours.measure(fit_rows), contours.measure(test_rows)
truth = test_rows["perspective"].to_numpy()

variants = {
    "contour features": dict(use_mirrored=False, use_subcategory=False),
    "plus mirrored copies as extra training images": dict(use_mirrored=True, use_subcategory=False),
    "plus the product subcategory": dict(use_mirrored=True, use_subcategory=True),
}
results = {"constant answer": {"macro F1": round(f1_score(truth, np.full(len(truth), "front_head_on"), average="macro"), 3), "accuracy": round(float((truth == "front_head_on").mean()), 3)}}
for label, options in variants.items():
    view_model = contours.PerspectiveModel(**options).fit(fit_rows, fit_features)
    predicted = view_model.predict(test_rows, test_features)
    results[label] = {"macro F1": round(f1_score(truth, predicted, average="macro"), 3), "accuracy": round(float((truth == predicted).mean()), 3)}
print(f"trained on {len(fit_rows)} images, tested on {len(test_rows)} holdout images")
display(pd.DataFrame(results).T)

side = lambda label: "left" if label in ("front_angle_left", "left_side", "back_angle_right") else "right"
sided = np.isin(truth, list(contours.MIRRORED_CLASS))
called = sided & np.isin(predicted, list(contours.MIRRORED_CLASS))
print(f"left or right: {int(sided.sum())} holdout images have a left or right label. The model also names a side for {int(called.sum())} of them "
      f"and picks the correct side for {np.mean([side(a) == side(b) for a, b in zip(truth[called], predicted[called])]):.2f} of those.")
print("\nshare of the decisions carried by each group of features")
display(view_model.importance_by_group().round(2).to_frame("share"))
contour_view = pd.Series(predicted, index=test_rows["image_id"])

trained on 5990 images, tested on 1562 holdout images


,macro F1,accuracy
constant answer,0.049,0.327
contour features,0.478,0.565
plus mirrored copies as extra training images,0.501,0.585
plus the product subcategory,0.527,0.609


left or right: 679 holdout images have a left or right label. The model also names a side for 397 of them and picks the correct side for 0.83 of those.

share of the decisions carried by each group of features


,share
silhouette,0.29
edge directions,0.29
brightness,0.27
subcategory,0.10
summary,0.05
shadow,0.01


The same comparison with the leaderboard metric on the two unseen sets: the perspective that Apertus answers in the
tree against the perspective of the contour model.

In [13]:
for name in ("test", "benchmark"):
    with_tree = evaluate.tree_predictions(answers[name], backdrops[name])
    table = evaluate.score_table({"tree, perspective asked": with_tree, "tree, perspective from contours": evaluate.with_perspective(with_tree, contour_view)}, sets[name])
    print(f"\n{name}")
    display(table[["images", "perspective", "overall", "weighted"]])


test


,images,perspective,overall,weighted
"tree, perspective asked",300,0.146,0.559,0.751
"tree, perspective from contours",300,0.447,0.609,0.790



benchmark


,images,perspective,overall,weighted
"tree, perspective asked",500,0.151,0.547,0.754
"tree, perspective from contours",500,0.526,0.609,0.798


The contour features were also tried on the other five dimensions, with colour and the subcategory added. They only
help where the answer is a matter of geometry. The last column is the tree on the benchmark images for comparison.

In [14]:
all_features = contours.measure(train)
in_holdout = holdout.to_numpy()
subcategory = pd.get_dummies(train["subcategory"]).to_numpy(dtype=np.float32)
inputs = {
    "contours": all_features["shape"],
    "contours, colour, subcategory": np.hstack([all_features["shape"], all_features["colour"], subcategory]),
}
scores = {label: {} for label in inputs}
for dimension in DIMENSIONS:
    scored = data.is_scored(train, dimension).to_numpy()
    labels = train[dimension].to_numpy()
    for label, table in inputs.items():
        forest = ExtraTreesClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=SEED).fit(table[scored & ~in_holdout], labels[scored & ~in_holdout])
        scores[label][dimension] = round(f1_score(labels[scored & in_holdout], forest.predict(table[scored & in_holdout]), average="macro"), 3)
tree_scores = evaluate.score_table({"tree": evaluate.tree_predictions(answers["benchmark"], backdrops["benchmark"])}, sets["benchmark"]).loc["tree"]
comparison = pd.DataFrame(scores)
comparison["question tree, benchmark images"] = [tree_scores[d] for d in comparison.index]
comparison

,contours,"contours, colour, subcategory","question tree, benchmark images"
subject,0.375,0.391,0.483
background,0.597,0.726,0.733
perspective,0.469,0.500,0.151
individuals,0.507,0.608,0.929
text_classification,0.318,0.349,0.476
text_language,0.284,0.302,0.508


## 9. A learned model instead of the hand-written tree

`question_tree.combiner` gets the same answers and the same backdrop measurement as the tree and learns from the
tuning images how to combine them. It is scored on the two unseen sets, with the perspective from the contour model in
both rows. It is ahead of the tree on one set and behind on the other, where its language answers break down. The
hand-written tree therefore stays the default, and the combiner is an option that needs more labeled images.

In [15]:
learned = combiner.Combiner().fit(answers["tune"][TREE_QUESTIONS], backdrops["tune"], sets["tune"])
final = {}
for name in ("test", "benchmark"):
    by_tree = evaluate.with_perspective(evaluate.tree_predictions(answers[name], backdrops[name]), contour_view)
    by_model = evaluate.with_perspective(learned.predict(answers[name][TREE_QUESTIONS], backdrops[name]), contour_view)
    final[name] = {"tree with sharpened rules, perspective from contours": by_tree, "learned combiner, perspective from contours": by_model}
    print(f"\n{name}")
    display(evaluate.score_table(final[name], sets[name]))


test


,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
"tree with sharpened rules, perspective from contours",300,0.432,0.770,0.447,0.927,0.446,0.635,0.609,0.79
"learned combiner, perspective from contours",300,0.474,0.764,0.447,0.968,0.550,0.357,0.593,0.81



benchmark


,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
"tree with sharpened rules, perspective from contours",500,0.483,0.733,0.526,0.929,0.476,0.508,0.609,0.798
"learned combiner, perspective from contours",500,0.476,0.766,0.526,0.969,0.572,0.487,0.633,0.806


## 10. Fewer requests: questions that share a request

The tree asks 22 yes/no questions and 3 multiple-choice questions, one per request. Putting all 22 yes/no questions
into one request was tried on the tuning images. The first table scores the tree on the images whose grouped reply came
back complete, with the perspective left out because it does not depend on these questions. The second table shows,
per question, how often the grouped answer equals the answer given alone.

In [16]:
def read_grouped(file_name):
    records = [json.loads(line) for line in (STORE / file_name).read_text().splitlines()]
    return pd.DataFrame(records).pivot(index="image_id", columns="key", values="answer")


def five_dimensions(predictions, rows):
    result = evaluate.evaluate(predictions, rows)
    scores = {d: round(result[d]["f1_macro"], 3) for d in DIMENSIONS if d != "perspective"}
    return {**scores, "mean of the five": round(float(np.mean(list(scores.values()))), 3)}


grouped = read_grouped(f"grouped_all22_tune_{MODEL_8B}.jsonl")
alone = answers["tune"].loc[grouped.index]
together = alone.copy()
together[TREE_YES_NO] = grouped[TREE_YES_NO]
rows = sets["tune"][sets["tune"]["image_id"].isin(grouped.index)]
print(f"{len(grouped)} of {len(sets['tune'])} grouped replies were complete")
display(pd.DataFrame({
    "every question alone, 25 requests": five_dimensions(evaluate.tree_predictions(alone, backdrops["tune"]), rows),
    "22 yes/no questions in one request, 4 requests": five_dimensions(evaluate.tree_predictions(together, backdrops["tune"]), rows),
}).T)
agreement = pd.Series({key: round(float((alone[key] == grouped[key]).mean()), 2) for key in TREE_YES_NO}).sort_values()
agreement.to_frame("same answer as when asked alone")

469 of 474 grouped replies were complete


,subject,background,individuals,text_classification,text_language,mean of the five
"every question alone, 25 requests",0.548,0.772,0.968,0.525,0.590,0.681
"22 yes/no questions in one request, 4 requests",0.289,0.755,0.962,0.338,0.579,0.585


,same answer as when asked alone
packaging,0.36
pack_and_real,0.59
promo,0.79
several,0.80
infographic,0.81
photo,0.86
plain,0.86
words2,0.88
scope,0.90
seal,0.91


The seven questions whose answers change most are therefore always asked alone, and the other fifteen share one
request (`ASKED_ALONE` and `ASKED_TOGETHER` in `question_tree.questions`). This was run for real on the benchmark
images. A grouped reply sometimes stalls before its end. The answers it holds until then are used, and a question
without an answer falls back to the answer given alone. The switch is `GROUP_ROBUST_QUESTIONS` in `question_tree.config`.

In [17]:
name, rows = "benchmark", sets["benchmark"]
grouped = read_grouped(f"grouped_mixed_benchmark_{MODEL_8B}.jsonl").reindex(rows["image_id"])
mixed = answers[name].copy()
fallbacks = 0
for key in ASKED_TOGETHER:
    has_answer = grouped[key].isin(["yes", "no"])
    mixed.loc[has_answer.to_numpy(), key] = grouped[key][has_answer].to_numpy()
    fallbacks += int((~has_answer).sum())
requests_alone = len(TREE_QUESTIONS)
requests_mixed = 1 + len(ASKED_ALONE) + len(TREE_MULTIPLE_CHOICE) + fallbacks / len(rows)
print(f"grouped replies that stalled: {int((grouped['_stalled'] == 'yes').sum())} of {len(rows)} | questions that fell back to being asked alone: {fallbacks} of {len(rows) * len(ASKED_TOGETHER)}")
table = evaluate.score_table({
    "every question alone": evaluate.with_perspective(evaluate.tree_predictions(answers[name], backdrops[name]), contour_view),
    "15 questions share one request, 7 alone": evaluate.with_perspective(evaluate.tree_predictions(mixed, backdrops[name]), contour_view),
}, rows)
table.insert(1, "requests per image", [requests_alone, round(requests_mixed, 1)])
table

grouped replies that stalled: 26 of 500 | questions that fell back to being asked alone: 29 of 7500


,images,requests per image,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
every question alone,500,25.0,0.483,0.733,0.526,0.929,0.476,0.508,0.609,0.798
"15 questions share one request, 7 alone",500,11.1,0.459,0.733,0.526,0.925,0.458,0.508,0.602,0.795


## 11. What was tried and dropped

**Other ways of prompting.** Three strategies with one or two requests per image, on the 200 test images they were run
on: the complete labeling guideline as instructions, the thinking variant of the model, and letting the model describe
the image first and label it from its own description. The baseline and the tree are scored on the same images.

In [18]:
def strategy(name, model=MODEL_8B):
    return evaluate.read_strategy(STORE / f"strategy_{name}_test_{model}.jsonl")


rows = sets["test"]
strategies = {
    "one prompt for all six labels": baseline,
    "full guideline as instructions": strategy("full_guideline"),
    "thinking variant": strategy("thinking"),
    "describe, then classify the description": strategy("describe_then_classify"),
    "tree with sharpened rules": evaluate.tree_predictions(answers["test"], backdrops["test"]),
}
display(evaluate.score_table(strategies, rows))

shared_ids = set(strategies["full guideline as instructions"]["image_id"])
compared = rows[rows["image_id"].isin(shared_ids)]
short_prompt = baseline.set_index("image_id").loc[compared["image_id"]]
long_prompt = strategies["full guideline as instructions"].set_index("image_id").loc[compared["image_id"]]
cells = {}
for dimension in DIMENSIONS:
    scored = data.is_scored(compared, dimension).to_numpy()
    right_before = (short_prompt[dimension].to_numpy() == compared[dimension].to_numpy())[scored]
    right_after = (long_prompt[dimension].to_numpy() == compared[dimension].to_numpy())[scored]
    cells[dimension] = {"scored cells": int(scored.sum()), "right with the short prompt": int(right_before.sum()), "right with the full guideline": int(right_after.sum()),
                        "fixed": int((~right_before & right_after).sum()), "broken": int((right_before & ~right_after).sum())}
print("the full guideline against the short prompt, cell by cell")
display(pd.DataFrame(cells).T)

effort = {}
for label, model in (("8B", MODEL_8B), ("70B", MODEL_70B)):
    records = pd.DataFrame([json.loads(line) for line in (STORE / f"strategy_thinking_test_{model}.jsonl").read_text().splitlines()])
    effort[f"thinking variant, {label}"] = {"images": len(records), "images with reasoning text": int((records["reasoning_chars"] > 0).sum()),
                                           "median tokens written": float(records["completion_tokens"].median()), "median seconds": float(records["seconds"].median())}
pd.DataFrame(effort).T.astype({"images": int, "images with reasoning text": int})

,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
one prompt for all six labels,200,0.054,0.464,0.114,0.568,0.185,0.612,0.333,0.631
full guideline as instructions,200,0.054,0.358,0.076,0.568,0.272,0.308,0.273,0.587
thinking variant,200,0.096,0.616,0.261,0.995,0.362,0.509,0.473,0.640
"describe, then classify the description",200,0.190,0.574,0.366,0.910,0.426,0.700,0.528,0.684
tree with sharpened rules,200,0.427,0.789,0.181,0.955,0.436,0.651,0.573,0.746


the full guideline against the short prompt, cell by cell


,scored cells,right with the short prompt,right with the full guideline,fixed,broken
subject,189,102,102,0,0
background,166,118,105,6,19
perspective,124,37,31,1,7
individuals,166,156,156,0,0
text_classification,184,158,151,5,12
text_language,79,59,41,9,27


,images,images with reasoning text,median tokens written,median seconds
"thinking variant, 8B",200,200,276.0,2.085
"thinking variant, 70B",200,0,53.0,1.660


**The larger model.** The 70B answered the questions of the first tree on 200 of the test images. Both models are
scored on those images with the same tree. The thinking variant of the 70B returned no reasoning, as the table above
shows, so thinking was only tested on the 8B.

In [19]:
answers_70b = ask.read_answers(STORE / f"answers_test_{MODEL_70B}.jsonl")
rows = sets["test"][sets["test"]["image_id"].isin(answers_70b.index)]
answers_70b = answers_70b.reindex(rows["image_id"])
evaluate.score_table({
    "8B, one prompt for all six labels": baseline,
    "8B, tree, first version": evaluate.tree_predictions(answers["test"].loc[rows["image_id"]], backdrops["test"], tree.NO_RULES),
    "70B, one prompt for all six labels": evaluate.read_baseline(MODEL_70B, RUNS_DIR),
    "70B, tree, first version": evaluate.tree_predictions(answers_70b, backdrops["test"], tree.NO_RULES),
}, rows)

,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
"8B, one prompt for all six labels",200,0.054,0.464,0.114,0.568,0.185,0.612,0.333,0.631
"8B, tree, first version",200,0.433,0.789,0.181,0.955,0.436,0.630,0.571,0.746
"70B, one prompt for all six labels",200,0.054,0.427,0.092,0.640,0.306,0.267,0.298,0.611
"70B, tree, first version",200,0.429,0.788,0.084,0.935,0.392,0.577,0.534,0.714


**Understanding the object first.** To find accessories, the model named the object, said what it is used for, and
then decided whether the image shows the listed product, a part of it, or the product with extras
(`experiments.role_chain`). An image that the tree calls full_product became accessories when the chain said "part"
and the model also called it a part of a bigger product. The table counts what that rule changes on the benchmark
images.

In [20]:
name, rows = "benchmark", sets["benchmark"]
by_tree = evaluate.tree_predictions(answers[name], backdrops[name])
says_part = ((answers[name]["role_b"] == "part") & (answers[name]["part_of"] == "yes")).to_numpy()
with_rule = by_tree.copy()
with_rule.loc[(by_tree["subject"] == "full_product").to_numpy() & says_part, "subject"] = "accessories"
scored = data.is_scored(rows, "subject").to_numpy()
changed = scored & (by_tree["subject"] != with_rule["subject"]).to_numpy()
label = rows["subject"].to_numpy()
print(f"accessories among the scored benchmark images: {int((label[scored] == 'accessories').sum())}")
print(f"answers changed by the rule: {int(changed.sum())} | fixed: {int((changed & (with_rule['subject'].to_numpy() == label)).sum())} | broken: {int((changed & (by_tree['subject'].to_numpy() == label)).sum())}")
evaluate.score_table({"tree": by_tree, "tree with the accessory rule": with_rule}, rows)[["images", "subject", "overall", "weighted"]]

accessories among the scored benchmark images: 4
answers changed by the rule: 38 | fixed: 3 | broken: 28


,images,subject,overall,weighted
tree,500,0.483,0.547,0.754
tree with the accessory rule,500,0.490,0.548,0.748


**Grayscale images.** Two multiple-choice questions were asked again on a grayscale copy of every tuning image. The
answers barely change.

In [21]:
name, rows, gray = "tune", sets["tune"], {}
for key, dimension in (("subject_mc", "subject"), ("persp_mc", "perspective")):
    scored = data.is_scored(rows, dimension).to_numpy() & answers[name][f"{key}_gray"].notna().to_numpy()
    label = rows[dimension].to_numpy()[scored]
    colour_answer, gray_answer = answers[name][key].to_numpy()[scored], answers[name][f"{key}_gray"].to_numpy()[scored]
    gray[dimension] = {"scored cells": int(scored.sum()), "accuracy on the colour image": round(float((colour_answer == label).mean()), 3),
                       "accuracy on the grayscale image": round(float((gray_answer == label).mean()), 3), "same answer on both": round(float((colour_answer == gray_answer).mean()), 2)}
pd.DataFrame(gray).T.astype({"scored cells": int})

,scored cells,accuracy on the colour image,accuracy on the grayscale image,same answer on both
subject,431,0.538,0.552,0.9
perspective,310,0.326,0.332,0.9


## 12. The pipeline, step by step

The leaderboard score and the weighted F1 on the two unseen image sets, from the one-prompt baseline to the full
pipeline. The last table lists the speeds that were measured while the answers were collected through the shared
Apertus endpoint. They depend on its load at the time and are not recomputed here.

In [22]:
for name in ("test", "benchmark"):
    rows, steps = sets[name], methods(name)
    steps["tree with sharpened rules, perspective from contours"] = final[name]["tree with sharpened rules, perspective from contours"]
    steps["learned combiner, perspective from contours"] = final[name]["learned combiner, perspective from contours"]
    print(f"\n{name}")
    display(evaluate.score_table(steps, rows))
pd.DataFrame(json.loads((STORE / "timings.json").read_text())).astype(object).map(lambda value: "" if pd.isna(value) else value)


test

,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
constant answer,300,0.056,0.302,0.042,0.481,0.185,0.102,0.195,0.578
one prompt for all six labels,300,0.056,0.454,0.095,0.535,0.185,0.600,0.321,0.631
one multiple-choice question per dimension,300,0.417,0.494,0.132,0.927,0.410,0.611,0.498,0.719
"tree, first version",300,0.431,0.770,0.146,0.927,0.447,0.611,0.555,0.747
tree with sharpened rules,300,0.432,0.770,0.146,0.927,0.446,0.635,0.559,0.751
"tree with sharpened rules, perspective from contours",300,0.432,0.770,0.447,0.927,0.446,0.635,0.609,0.790
"learned combiner, perspective from contours",300,0.474,0.764,0.447,0.968,0.550,0.357,0.593,0.810



benchmark


,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
constant answer,500,0.058,0.308,0.047,0.489,0.152,0.072,0.188,0.592
one prompt for all six labels,500,0.058,0.545,0.070,0.546,0.150,0.352,0.287,0.634
one multiple-choice question per dimension,500,0.389,0.544,0.112,0.929,0.484,0.484,0.490,0.717
"tree, first version",500,0.431,0.733,0.151,0.929,0.486,0.484,0.536,0.743
tree with sharpened rules,500,0.483,0.733,0.151,0.929,0.476,0.508,0.547,0.754
"tree with sharpened rules, perspective from contours",500,0.483,0.733,0.526,0.929,0.476,0.508,0.609,0.798
"learned combiner, perspective from contours",500,0.476,0.766,0.526,0.969,0.572,0.487,0.633,0.806


,what,requests_per_image,parallel_images,images_per_second,requests_per_second,images,seconds
0,"tree, every question alone, 8B",24.0,24.0,0.87,17.5,300.0,
1,"tree, every question alone, 70B",24.0,4.0,0.18,2.9,200.0,
2,"the shared request of 15 questions on its own, 8B",1.0,24.0,7.1,,500.0,
3,"tree with grouping, 8B, dev images",11.0,24.0,1.35,,250.0,
4,"tree with grouping, 8B, dev images",11.0,48.0,1.48,,250.0,
5,"tree with grouping, 8B, team holdout",11.0,24.0,1.64,,600.0,
6,"first question on an image, 8B, idle gateway",,,,,,0.77
7,"each further question on the same image, 8B, idle gateway",,,,,,0.13


## 13. One holdout for the whole team

The trained methods in this repository, the classification heads and the LoRA finetune, hold out the 600 images of the
reference notebook (`common.split_holdout`) and train on all other images. Those other images include most of the test
and benchmark images used above, so the two sets cannot score a trained method. The question tree needs no training,
so it is scored here on the 600-image holdout instead. The tuning images of the tree that lie in this holdout are left
out, and the contour model is trained again on the labeled images outside it. The answers were collected with the
grouped requests of section 10, as the pipeline is configured. The last row can be compared directly with the holdout
scores that `heads.py` and `lora_atomic.py` print.

In [23]:
reference = data.reference_holdout(train)
used_for_tuning = reference["image_id"].isin(sets["tune"]["image_id"])
display(pd.Series({
    "images in the holdout of the trained methods": len(reference),
    "of them tuning images of the tree, left out here": int(used_for_tuning.sum()),
    "test images that the trained methods train on": int((~sets["test"]["image_id"].isin(reference["image_id"])).sum()),
    "benchmark images that the trained methods train on": int((~sets["benchmark"]["image_id"].isin(reference["image_id"])).sum()),
}).to_frame("images"))

unseen = reference[~used_for_tuning].reset_index(drop=True)
reference_answers = ask.read_answers(STORE / f"answers_reference_holdout_{MODEL_8B}.jsonl").reindex(unseen["image_id"])
by_tree = evaluate.tree_predictions(reference_answers, evaluate.measure_backdrops(unseen))
outside = labeled[~labeled["product_id"].isin(set(reference["product_id"]))]
view_model = contours.PerspectiveModel().fit(outside, contours.measure(outside))
view = pd.Series(view_model.predict(unseen, contours.measure(unseen)), index=unseen["image_id"])
print(f"contour model trained on {len(outside)} labeled images outside this holdout")
evaluate.score_table({
    "constant answer": evaluate.constant_predictions(unseen),
    "one prompt for all six labels": baseline,
    "question tree, perspective from contours": evaluate.with_perspective(by_tree, view),
}, unseen)

,images
images in the holdout of the trained methods,600
"of them tuning images of the tree, left out here",27
test images that the trained methods train on,288
benchmark images that the trained methods train on,473


contour model trained on 7194 labeled images outside this holdout


,images,subject,background,perspective,individuals,text_classification,text_language,overall,weighted
constant answer,573,0.053,0.302,0.053,0.487,0.153,0.079,0.188,0.578
one prompt for all six labels,573,0.053,0.442,0.122,0.525,0.150,0.560,0.309,0.622
"question tree, perspective from contours",573,0.427,0.678,0.552,0.882,0.438,0.540,0.586,0.776
